In [ ]:
# INHARD - R3D_18(Train)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models.video as models
from torchvision.models.video import R3D_18_Weights
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import glob

BATCH_SIZE = 8
EPOCHS = 25
LEARNING_RATE = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

action_classes = sorted(os.listdir("/nfsshare/knr/Havid/RGBSeg/"))
action_to_label = {action: idx for idx, action in enumerate(action_classes)}

class InHARDVideoDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_video(video_path)
        return frames, label  
    
    def load_video(self, path, num_frames=16):
        cap = cv2.VideoCapture(path)
        frames = []
    
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (112, 112))  
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
    
        cap.release()
    
        frames = np.array(frames, dtype=np.float32) / 255.0  
    
        # Ensure enough frames
        if len(frames) < num_frames:
            pad = np.zeros((num_frames - len(frames), 112, 112, 3), dtype=np.float32)
            frames = np.concatenate((frames, pad), axis=0)
        else:
            frames = frames[:num_frames]
    
        frames = np.transpose(frames, (3, 0, 1, 2))  
        return torch.tensor(frames, dtype=torch.float32)

video_paths = glob.glob("/nfsshare/knr/Havid/RGBSeg/*/*.mp4")
labels = [action_to_label[os.path.basename(os.path.dirname(p))] for p in video_paths]

dataset = InHARDVideoDataset(video_paths, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = models.r3d_18(weights=R3D_18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(action_classes))  
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for videos, labels in dataloader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(dataloader):.4f}, Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), "/nfsshare/knr/Havid/inhard_r3d18.pth")
print("Model fine-tuned and saved successfully!")


In [ ]:
# INHARD - R3D_18(Test)

import torch
import torch.nn as nn
import cv2
import numpy as np
from collections import deque
import time
import torchvision.models.video as models

action_classes = ["Assemble system", "Consult sheets", "No action", "Picking in front", 
                  "Picking left", "Put down component", "Put down measuring rod", "Put down screwdriver", 
                  "Put down subsystem", "Take component", "Take measuring rod", "Take screwdriver", 
                  "Take subsystem", "Turn sheets"]

def test_unsegmented_video(video_path, model_path, action_classes, device="cuda"):
    model = models.r3d_18(weights=None) 
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, len(action_classes))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_delay = int(1000 / fps) if fps > 0 else 40  
    frame_buffer = deque(maxlen=16)
    
    prev_time = time.time()
    fps_display = fps if fps > 0 else 30 
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (112, 112))
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)
        
        predicted_action = "Detecting..."
        if len(frame_buffer) == 16:
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            frames = np.transpose(frames, (3, 0, 1, 2))  # (C, T, H, W)
            frames_tensor = torch.tensor(frames, dtype=torch.float32).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = model(frames_tensor)
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time
        
        overlay_text = [
            f"Action: {predicted_action}",
            f"FPS: {fps_display:.2f}"  
        ]

        for i, text in enumerate(overlay_text):
            cv2.putText(frame, text, (30, 500 + i * 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3, cv2.LINE_AA)
        
        cv2.imshow("Real-Time Action Recognition", frame)

        if cv2.waitKey(frame_delay) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

video_path = r"C:\Users\HP\OneDrive\Desktop\har\P01_R01.mp4"
model_path = r"C:\Users\HP\Downloads\Models\inhard_r3d18.pth"
test_unsegmented_video(video_path, model_path, action_classes)


In [ ]:
# INHARD - Slowfast(Train)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from pytorchvideo.models.slowfast import create_slowfast
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import glob

BATCH_SIZE = 8
EPOCHS = 25
LEARNING_RATE = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

action_classes = sorted(os.listdir("/nfsshare/knr/Havid/RGBSeg/"))
action_to_label = {action: idx for idx, action in enumerate(action_classes)}

class InHARDVideoDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_video(video_path)
        return [f.to(device) for f in frames], torch.tensor(label, dtype=torch.long).to(device)

    def load_video(self, path, slow_num_frames=8, fast_num_frames=32):
        cap = cv2.VideoCapture(path)
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (224, 224))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        
        frames = np.array(frames, dtype=np.float32) / 255.0
        if len(frames) < fast_num_frames:
            pad = np.zeros((fast_num_frames - len(frames), 224, 224, 3), dtype=np.float32)
            frames = np.concatenate((frames, pad), axis=0)
        
        fast_pathway = frames[:fast_num_frames]
        slow_pathway = fast_pathway[::4][:slow_num_frames]
        
        slow_pathway = torch.tensor(np.transpose(slow_pathway, (3, 0, 1, 2)), dtype=torch.float32)
        fast_pathway = torch.tensor(np.transpose(fast_pathway, (3, 0, 1, 2)), dtype=torch.float32)
        
        return [slow_pathway, fast_pathway]

video_paths = glob.glob("/nfsshare/knr/Havid/RGBSeg/*/*.mp4")
labels = [action_to_label[os.path.basename(os.path.dirname(p))] for p in video_paths]

dataset = InHARDVideoDataset(video_paths, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = create_slowfast(model_num_class=14).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

all_preds, all_labels = [], []
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for videos, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(dataloader):.4f}, Accuracy: {acc:.2f}%")

torch.save(model.state_dict(), "/nfsshare/knr/Havid/inhard_slowfast.pth")
print("Model saved successfully!")


In [ ]:
# INHARD - Slowfast(Test)

import torch
import cv2
import numpy as np
from collections import deque
from pytorchvideo.models.slowfast import create_slowfast
import time

action_classes = ["Assemble system", "Consult sheets", "No action", "Picking in front", 
                  "Picking left", "Put down component", "Put down measuring rod", "Put down screwdriver", 
                  "Put down subsystem", "Take component", "Take measuring rod", "Take screwdriver", 
                  "Take subsystem", "Turn sheets"]

def test_unsegmented_video(video_path, model_path, action_classes, device="cuda"):
    model = create_slowfast(model_num_class=len(action_classes))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_delay = int(1000 / fps) if fps > 0 else 40  
    frame_buffer = deque(maxlen=32)

    prev_time = time.time()
    fps_display = fps if fps > 0 else 30  

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (224, 224))
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)

        predicted_action = "Detecting..."
        if len(frame_buffer) == 32:
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            fast_pathway = frames
            slow_pathway = fast_pathway[::4]

            slow_tensor = torch.tensor(np.transpose(slow_pathway, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)
            fast_tensor = torch.tensor(np.transpose(fast_pathway, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model([slow_tensor, fast_tensor])
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time

        overlay_text = [
            f"Action: {predicted_action}",
            f"FPS: {fps_display:.2f}" 
        ]

        for i, text in enumerate(overlay_text):
            cv2.putText(frame, text, (30, 500 + i * 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3, cv2.LINE_AA)

        cv2.imshow("Real-Time Action Recognition", frame)

        if cv2.waitKey(frame_delay) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

video_path = r"C:\Users\HP\OneDrive\Desktop\har\P01_R01.mp4"
model_path = r"C:\Users\HP\Downloads\Models\inhard_slowfast.pth"
test_unsegmented_video(video_path, model_path, action_classes)


In [ ]:
# KNR - R3D_18(Train)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models.video import r3d_18
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import glob

BATCH_SIZE = 8
EPOCHS = 25
LEARNING_RATE = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_path = r"C:\Users\HP\OneDrive\Desktop\Data"

action_classes = sorted(os.listdir(dataset_path))
action_to_label = {action: idx for idx, action in enumerate(action_classes)}

class CustomVideoDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_video(video_path)
        return frames.to(device), torch.tensor(label, dtype=torch.long).to(device)

    def load_video(self, path, num_frames=16):
        cap = cv2.VideoCapture(path)
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (112, 112))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        
        frames = np.array(frames, dtype=np.float32) / 255.0
        if len(frames) < num_frames:
            pad = np.zeros((num_frames - len(frames), 112, 112, 3), dtype=np.float32)
            frames = np.concatenate((frames, pad), axis=0)
        else:
            frames = frames[:num_frames]
        
        frames = torch.tensor(np.transpose(frames, (3, 0, 1, 2)), dtype=torch.float32)
        return frames  

video_paths = glob.glob(os.path.join(dataset_path, "*/*.mp4"))
labels = [action_to_label[os.path.basename(os.path.dirname(p))] for p in video_paths]

dataset = CustomVideoDataset(video_paths, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = r3d_18(weights="R3D_18_Weights.KINETICS400_V1")  
model.fc = nn.Linear(model.fc.in_features, len(action_classes))  
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

all_preds, all_labels = [], []
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for videos, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(videos.squeeze(1)) 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(dataloader):.4f}, Accuracy: {acc:.2f}%")

torch.save(model.state_dict(), r"C:\Users\HP\OneDrive\Desktop\Data\r3d18_model.pth")
print("Model saved successfully!")


In [ ]:
# KNR - R3D_18(Test)

import torch
import cv2
import numpy as np
from collections import deque
from torchvision.models.video import r3d_18, R3D_18_Weights
import time

action_classes = ["Checking", "Missing", "Picking", "Rejected", "Accepted"]

def test_unsegmented_video(video_path, model_path, action_classes, device="cuda"):
    model = r3d_18(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, len(action_classes))  
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_delay = int(1000 / fps) if fps > 0 else 40
    frame_buffer = deque(maxlen=16)  

    prev_time = time.time()
    fps_display = fps if fps > 0 else 30

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (112, 112))  
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)
        predicted_action = "Detecting..."
        if len(frame_buffer) == 16:  
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            frames = np.transpose(frames, (3, 0, 1, 2))  

            input_tensor = torch.tensor(frames, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model(input_tensor)
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time

        overlay_text = [
            f"Action: {predicted_action}",
            f"FPS: {fps_display:.2f}"
        ]

        for i, text in enumerate(overlay_text):
            cv2.putText(frame, text, (30, 500 + i * 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3, cv2.LINE_AA)

        cv2.imshow("Real-Time Action Recognition", frame)

        if cv2.waitKey(frame_delay) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

video_path = r"C:\Users\HP\OneDrive\Desktop\har\2.mp4"
model_path = r"C:\Users\HP\Downloads\Models\tab_r3d18.pth"
test_unsegmented_video(video_path, model_path, action_classes)


In [ ]:
# KNR - Slowfast(Train)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from pytorchvideo.models.slowfast import create_slowfast
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import glob

BATCH_SIZE = 4
EPOCHS = 25
LEARNING_RATE = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_path = "/nfsshare/knr/Havid/Data"

action_classes = sorted(os.listdir(dataset_path))
action_to_label = {action: idx for idx, action in enumerate(action_classes)}

class CustomVideoDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_video(video_path)
        return [f.to(device) for f in frames], torch.tensor(label, dtype=torch.long).to(device)

    def load_video(self, path, slow_num_frames=8, fast_num_frames=32):
        cap = cv2.VideoCapture(path)
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (224, 224))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        
        frames = np.array(frames, dtype=np.float32) / 255.0
        if len(frames) < fast_num_frames:
            pad = np.zeros((fast_num_frames - len(frames), 224, 224, 3), dtype=np.float32)
            frames = np.concatenate((frames, pad), axis=0)
        
        fast_pathway = frames[:fast_num_frames]
        slow_pathway = fast_pathway[::4][:slow_num_frames]
        
        slow_pathway = torch.tensor(np.transpose(slow_pathway, (3, 0, 1, 2)), dtype=torch.float32)
        fast_pathway = torch.tensor(np.transpose(fast_pathway, (3, 0, 1, 2)), dtype=torch.float32)
        
        return [slow_pathway, fast_pathway]

video_paths = glob.glob(os.path.join(dataset_path, "*/*.mp4"))
labels = [action_to_label[os.path.basename(os.path.dirname(p))] for p in video_paths]

dataset = CustomVideoDataset(video_paths, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = create_slowfast(model_num_class=len(action_classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

all_preds, all_labels = [], []
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for videos, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(dataloader):.4f}, Accuracy: {acc:.2f}%")

torch.save(model.state_dict(), "/nfsshare/knr/Havid/model.pth")
print("Model saved successfully!")


In [ ]:
# KNR - Slowfast(Test)

import torch
import cv2
import numpy as np
from collections import deque
from pytorchvideo.models.slowfast import create_slowfast
import time

action_classes = ["Checking", "Missing", "Picking", "Rejected", "Accepted"]

def test_unsegmented_video(video_path, model_path, action_classes, device="cuda"):
    model = create_slowfast(model_num_class=len(action_classes))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_delay = int(1000 / fps) if fps > 0 else 40
    frame_buffer = deque(maxlen=32)
    
    prev_time = time.time()
    fps_display = fps if fps > 0 else 30

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (224, 224))
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)

        predicted_action = "Detecting..."
        if len(frame_buffer) == 32:
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            fast_pathway = frames
            slow_pathway = fast_pathway[::4]

            slow_tensor = torch.tensor(np.transpose(slow_pathway, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)
            fast_tensor = torch.tensor(np.transpose(fast_pathway, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model([slow_tensor, fast_tensor])
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time

        overlay_text = [
            f"Action: {predicted_action}",
            f"FPS: {fps_display:.2f}"
        ]

        for i, text in enumerate(overlay_text):
            cv2.putText(frame, text, (30, 500 + i * 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3, cv2.LINE_AA)

        cv2.imshow("Real-Time Action Recognition", frame)

        if cv2.waitKey(frame_delay) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

video_path = r"C:\Users\HP\OneDrive\Desktop\har\3.mp4"
model_path = r"C:\Users\HP\Downloads\Models\tab_slowfast.pth"
test_unsegmented_video(video_path, model_path, action_classes)


In [ ]:
# TVS - R3D_18(Train)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models.video import r3d_18
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import glob
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

BATCH_SIZE = 8
EPOCHS = 25
LEARNING_RATE = 0.001
NUM_FRAMES = 16  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_path = "/nfsshare/knr/Havid/TvsData"

action_classes = sorted(os.listdir(dataset_path))
action_to_label = {action: idx for idx, action in enumerate(action_classes)}

class CustomFrameDataset(Dataset):
    def __init__(self, root_dir, action_classes, num_frames=16, transform=None):
        self.root_dir = root_dir
        self.action_classes = action_classes
        self.num_frames = num_frames
        self.transform = transform
        self.data = self._load_data()

    def _load_data(self):
        data = []
        for action in self.action_classes:
            action_path = os.path.join(self.root_dir, action)
            for clip in os.listdir(action_path):
                clip_path = os.path.join(action_path, clip)
                frame_paths = sorted(glob.glob(os.path.join(clip_path, "*.png")))  
                if frame_paths:
                    data.append((frame_paths, self.action_classes.index(action)))
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        frame_paths, label = self.data[idx]
        frames = self.load_frames(frame_paths)
        return frames.to(device), torch.tensor(label, dtype=torch.long).to(device)

    def load_frames(self, frame_paths):
        frames = []
        for path in frame_paths[:self.num_frames]:  
            frame = cv2.imread(path)
            frame = cv2.resize(frame, (112, 112))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        
        frames = np.array(frames, dtype=np.float32) / 255.0
        if len(frames) < self.num_frames:
            pad = np.zeros((self.num_frames - len(frames), 112, 112, 3), dtype=np.float32)
            frames = np.concatenate((frames, pad), axis=0)

        frames = torch.tensor(np.transpose(frames, (3, 0, 1, 2)), dtype=torch.float32)
        return frames

dataset = CustomFrameDataset(dataset_path, action_classes, num_frames=NUM_FRAMES)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = r3d_18(weights="R3D_18_Weights.KINETICS400_V1")  
model.fc = nn.Linear(model.fc.in_features, len(action_classes))  
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    all_preds, all_labels = [], []

    for videos, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(dataloader):.4f}, Accuracy: {acc:.2f}%")

torch.save(model.state_dict(), r"/nfsshare/knr/Havid/r3d18_model_tvs.pth")
print("Model saved successfully!")


In [ ]:
# TVS - R3D_18(Test)

import torch
import cv2
import numpy as np
from collections import deque
import time
from torchvision.models.video import r3d_18

action_classes = ["Fixingbackscrew", "Fixing frame", "Fixingfrontscrew", 
                  "Fixingmiddlescrew", "Gettingtheframe", "Pickingscrews"]

def test_unsegmented_video(video_path, model_path, action_classes, device="cuda"):
    model = r3d_18(weights=None)  
    model.fc = torch.nn.Linear(model.fc.in_features, len(action_classes)) 
    model.load_state_dict(torch.load(model_path, map_location=device))  
    model.to(device).eval()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_delay = int(1000 / fps) if fps > 0 else 40
    frame_buffer = deque(maxlen=16)  

    prev_time = time.time()
    fps_display = fps if fps > 0 else 30

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (112, 112))
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)

        predicted_action = "Detecting..."
        if len(frame_buffer) == 16:
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            frames = torch.tensor(np.transpose(frames, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model(frames)
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time

        display_frame = cv2.resize(frame, (700, 500))

        overlay_height = 40
        overlay = np.zeros((overlay_height, 700, 3), dtype=np.uint8)

        text_color = (0, 255, 0)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.8
        thickness = 2
        offset = 30

        cv2.putText(overlay, f"Action: {predicted_action}", (30, offset), font, font_scale, text_color, thickness, cv2.LINE_AA)
        cv2.putText(overlay, f"FPS: {fps_display:.2f}", (400, offset), font, font_scale, text_color, thickness, cv2.LINE_AA)

        combined_frame = np.vstack((display_frame, overlay))

        cv2.imshow("Action Recognition", combined_frame)

        if cv2.waitKey(frame_delay) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

video_path = r"C:\Users\HP\Downloads\20221221_104535(1).mp4"
model_path = r"C:\Users\HP\Downloads\Models\tvs_r3d18.pth"
test_unsegmented_video(video_path, model_path, action_classes)


In [ ]:
# TVS - Slowfast(Train)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from pytorchvideo.models.slowfast import create_slowfast
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import glob

BATCH_SIZE = 8
EPOCHS = 25
LEARNING_RATE = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATASET_PATH = "/nfsshare/knr/Havid/Tvsdata/"

action_classes = sorted(os.listdir(DATASET_PATH))  
action_to_label = {action: idx for idx, action in enumerate(action_classes)}

class FrameDataset(Dataset):
    def __init__(self, dataset_path, transform=None, slow_num_frames=8, fast_num_frames=32):
        self.dataset_path = dataset_path
        self.transform = transform
        self.slow_num_frames = slow_num_frames
        self.fast_num_frames = fast_num_frames
        self.video_paths = []
        self.labels = []

        # Collect all frame folders
        for action in action_classes:
            action_folder = os.path.join(dataset_path, action)
            subfolders = sorted(os.listdir(action_folder))  
            for subfolder in subfolders:
                frame_folder = os.path.join(action_folder, subfolder)
                self.video_paths.append(frame_folder)
                self.labels.append(action_to_label[action])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        frame_folder = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_frames(frame_folder)
        return frames, label

    def load_frames(self, folder_path):
        frame_files = sorted(glob.glob(os.path.join(folder_path, "*.png")))  
        frames = []

        for frame_file in frame_files:
            frame = cv2.imread(frame_file)
            frame = cv2.resize(frame, (224, 224))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        frames = np.array(frames, dtype=np.float32) / 255.0  

        if len(frames) < self.fast_num_frames:
            pad = np.zeros((self.fast_num_frames - len(frames), 224, 224, 3), dtype=np.float32)
            frames = np.concatenate((frames, pad), axis=0)

        fast_pathway = frames[:self.fast_num_frames]

        slow_pathway = fast_pathway[::4][:self.slow_num_frames]

        slow_pathway = torch.tensor(np.transpose(slow_pathway, (3, 0, 1, 2)), dtype=torch.float32) 
        fast_pathway = torch.tensor(np.transpose(fast_pathway, (3, 0, 1, 2)), dtype=torch.float32)   

        return [slow_pathway, fast_pathway]  


transform = transforms.Compose([transforms.Lambda(lambda x: x)])  
dataset = FrameDataset(DATASET_PATH, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = create_slowfast(model_num_class=6) 
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for videos, labels in dataloader:
        videos = [v.to(device) for v in videos]  
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)  
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(dataloader):.4f}, Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), "/nfsshare/knr/Havid/action_recognition_slowfast.pth")
print("Model saved successfully!")


In [ ]:
# TVS - Slowfast(Test)

import torch
import cv2
import numpy as np
from collections import deque
from pytorchvideo.models.slowfast import create_slowfast
import time

action_classes = ["Fixingbackscrew", "Fixing frame", "Fixingfrontscrew", 
                  "Fixingmiddlescrew", "Gettingtheframe", "Pickingscrews"]

def test_unsegmented_video(video_path, model_path, action_classes, device="cuda"):
    model = create_slowfast(model_num_class=len(action_classes))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_delay = int(1000 / fps) if fps > 0 else 40
    frame_buffer = deque(maxlen=32)
    
    prev_time = time.time()
    fps_display = fps if fps > 0 else 30

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (224, 224))
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)

        predicted_action = "Detecting..."
        if len(frame_buffer) == 32:
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            fast_pathway = frames
            slow_pathway = fast_pathway[::4]

            slow_tensor = torch.tensor(np.transpose(slow_pathway, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)
            fast_tensor = torch.tensor(np.transpose(fast_pathway, (3, 0, 1, 2)), dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model([slow_tensor, fast_tensor])
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time

        display_frame = cv2.resize(frame, (700, 500))

        overlay_height = 40
        overlay = np.zeros((overlay_height, 700, 3), dtype=np.uint8)

        text_color = (0, 255, 0)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.8
        thickness = 2
        offset = 30

        cv2.putText(overlay, f"Action: {predicted_action}", (30, offset), font, font_scale, text_color, thickness, cv2.LINE_AA)
        cv2.putText(overlay, f"FPS: {fps_display:.2f}", (400, offset), font, font_scale, text_color, thickness, cv2.LINE_AA)

        combined_frame = np.vstack((display_frame, overlay))

        cv2.imshow("Action Recognition", combined_frame)

        if cv2.waitKey(frame_delay) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

video_path = r"C:\Users\HP\Downloads\20221221_104535(1).mp4"
model_path = r"C:\Users\HP\Downloads\Models\tvs_slowfast.pth"
test_unsegmented_video(video_path, model_path, action_classes)

In [3]:
import torch
import cv2
import numpy as np
from collections import deque
from torchvision.models.video import r3d_18
import time

action_classes = ["Checking", "Missing", "Picking", "Rejected", "Accepted"]

def load_model(model_path, device="cuda"):
    model = r3d_18(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, len(action_classes))  
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()
    return model

def test_live_video(model, action_classes, device="cuda"):
    cap = cv2.VideoCapture(1) 
    cap.set(3, 640) 
    cap.set(4, 480)  

    frame_buffer = deque(maxlen=16)
    prev_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (112, 112))  
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        frame_buffer.append(frame_rgb)

        predicted_action = "Detecting..."
        if len(frame_buffer) == 16:  
            frames = np.array(frame_buffer, dtype=np.float32) / 255.0
            frames = np.transpose(frames, (3, 0, 1, 2))  
            input_tensor = torch.tensor(frames, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model(input_tensor)
                predicted_class = torch.argmax(outputs, dim=1).item()
                predicted_action = action_classes[predicted_class]

        curr_time = time.time()
        fps_display = 1 / (curr_time - prev_time)
        prev_time = curr_time

        cv2.putText(frame, f"Action: {predicted_action}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"FPS: {fps_display:.2f}", (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Live Action Recognition", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

model_path = r"C:\Users\HP\Downloads\Models\tab_r3d18.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(model_path, device)

test_live_video(model, action_classes, device)
